In [7]:
from trade_data_import import get_trade_data_by_hscode
from DATA.stock_invest_function import *

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

from trade_data_import import get_unique_hscode_list

hs_list = get_unique_hscode_list(db_info)
print("HS Code 개수:", len(hs_list))
print("예시:", hs_list[:5])

hs_code = "854232"

trade_df_all = get_trade_data_by_hscode(db_info, hs_code)
# print(trade_df_all.head())

# 예시 2️⃣: HS Code + indicator 지정
trade_df_exp = get_trade_data_by_hscode(db_info, hs_code, 'expDlr')

[INFO] 총 1096개의 고유 HS Code를 불러왔습니다.
HS Code 개수: 1096
예시: ['1201', '121120', '1212', '121221', '151550']
[INFO] HS Code=854232, Indicator=ALL → 1125행 불러옴
[INFO] HS Code=854232, Indicator=expDlr → 225행 불러옴


In [8]:
hs_list

['1201',
 '121120',
 '1212',
 '121221',
 '151550',
 '1515500000',
 '151590',
 '1515901000',
 '1518',
 '1604',
 '170199',
 '1703',
 '190110',
 '1902',
 '190230',
 '1902301010',
 '19049',
 '1905',
 '190590',
 '200599',
 '2005991000',
 '2008',
 '200830',
 '2008301000',
 '200899',
 '2009',
 '210111',
 '2103',
 '210390',
 '2103901030',
 '2103909030',
 '2103909090',
 '210500',
 '2106',
 '210690',
 '2106901090',
 '2106904010',
 '2202',
 '220299',
 '220600',
 '220890',
 '2208904000',
 '230910',
 '2402',
 '240220',
 '2404',
 '250100',
 '250300',
 '260300',
 '261800',
 '2707',
 '270730',
 '270750',
 '270799',
 '2710',
 '271012',
 '271019',
 '2711',
 '271113',
 '2711130000',
 '271119',
 '2713',
 '271320',
 '2803',
 '280300',
 '280461',
 '280700',
 '2807001010',
 '2811',
 '281290',
 '2815',
 '281512',
 '281520',
 '2821',
 '282110',
 '2821101000',
 '2825',
 '282520',
 '283324',
 '283691',
 '283711',
 '2841',
 '284190',
 '2841909000',
 '2901',
 '290121',
 '290122',
 '290124',
 '2902',
 '290220',
 '2

In [2]:
from sarima_forecast_trade import sarima_forecast_trade_value

sarima_df = sarima_forecast_trade_value(
    df=trade_df_exp,
    indicator='expDlr',
    horizon=24,
    sarima_kwargs={
        # 필요 시 탐색범위/옵션 추가 가능 (없으면 자동값)
        # 'p_values': (0,1,2),
        # 'd_values': (0,1),
        # 'q_values': (0,1,2),
        # 'P_values': (0,1),
        # 'D_values': (0,1),
        # 'Q_values': (0,1),
        # 'try_transforms': True,
    }
)
# 결과: index=future dates, column='sarima_expDlr'
print(sarima_df.head())

[메모리] forecast_sarima 실행 전: 391.59 MB
[메모리] find_best_sarima_params 실행 전: 391.60 MB

[메모리] find_best_sarima_params 실행 후: 418.33 MB (변화: +26.73 MB)
[메모리] forecast_sarima 실행 후: 398.67 MB (변화: +7.08 MB)
            sarima_expDlr
date                     
2025-10-31   8.623064e+09
2025-11-30   8.770857e+09
2025-12-31   9.460374e+09
2026-01-31   7.894087e+09
2026-02-28   8.034311e+09


In [3]:
from multi_model_trade_forecast import forecast_trade_multi_models

# trade_df_exp: (date, indicator='expDlr', value) 구조 (root_hs_code 칼럼이 있어도 무방)
fc_table = forecast_trade_multi_models(
    trade_df=trade_df_exp,     # 위 스크린샷의 DF
    indicator="expDlr",
    horizon=24,                 # 6개월 예측
    model_kwargs={
        # 필요시 각 모델별 하이퍼파라미터 전달 가능
        # "SARIMA": {"seasonal_period": 12},
        # "LSTM": {"lookback": 12, "epochs": 40, "batch_size": 16},
    },
)
print(fc_table.head())

[메모리] forecast_sarima 실행 전: 398.73 MB
[메모리] find_best_sarima_params 실행 전: 398.73 MB
[메모리] find_best_sarima_params 실행 후: 418.66 MB (변화: +19.92 MB)
[메모리] forecast_sarima 실행 후: 399.12 MB (변화: +0.39 MB)
[메모리] forecast_ets 실행 전: 399.12 MB


21:15:01 - cmdstanpy - INFO - Chain [1] start processing
21:15:01 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_ets 실행 후: 399.33 MB (변화: +0.21 MB)
[메모리] forecast_prophet 실행 전: 399.33 MB
[메모리] forecast_prophet 실행 후: 401.21 MB (변화: +1.89 MB)
[메모리] forecast_lstm 실행 전: 401.21 MB


[메모리] forecast_lstm 실행 후: 485.04 MB (변화: +83.83 MB)
[메모리] forecast_theta 실행 전: 485.04 MB
[메모리] forecast_theta 실행 후: 485.21 MB (변화: +0.17 MB)
            sarima_expDlr    ets_expDlr  prophet_expDlr   lstm_expDlr  \
date                                                                    
2025-10-31   8.623064e+09  8.606155e+09    6.235488e+09  8.201339e+09   
2025-11-30   8.770857e+09  8.788921e+09    6.154282e+09  8.326431e+09   
2025-12-31   9.460374e+09  9.212108e+09    6.382724e+09  8.316274e+09   
2026-01-31   7.894087e+09  7.876946e+09    5.366620e+09  8.270964e+09   
2026-02-28   8.034311e+09  8.493473e+09    5.402308e+09  8.183201e+09   

            theta_expDlr  
date                      
2025-10-31  8.688979e+09  
2025-11-30  8.354402e+09  
2025-12-31  8.237087e+09  
2026-01-31  7.324136e+09  
2026

In [4]:
from ensemble_trade_forecast import build_ensemble_columns

out = build_ensemble_columns(fc_table, indicator="expDlr")

out

,sarima_expDlr,ets_expDlr,prophet_expDlr,lstm_expDlr,theta_expDlr,avg5_expDlr,ensemble_expDlr
date,,,,,,,
2025-10-31,8.623064e+09,8.606155e+09,6.235488e+09,8.201339e+09,8.688979e+09,8.071005e+09,8.476852e+09
2025-11-30,8.770857e+09,8.788921e+09,6.154282e+09,8.326431e+09,8.354402e+09,8.078979e+09,8.483897e+09
2025-12-31,9.460374e+09,9.212108e+09,6.382724e+09,8.316274e+09,8.237087e+09,8.321714e+09,8.588490e+09
2026-01-31,7.894087e+09,7.876946e+09,5.366620e+09,8.270964e+09,7.324136e+09,7.346551e+09,7.698390e+09
2026-02-28,8.034311e+09,8.493473e+09,5.402308e+09,8.183201e+09,7.277849e+09,7.478228e+09,7.831787e+09
2026-03-31,9.371595e+09,9.872599e+09,6.737653e+09,8.032862e+09,8.810172e+09,8.564976e+09,8.738210e+09
2026-04-30,8.440659e+09,8.961931e+09,5.824198e+09,7.857258e+09,8.106827e+09,7.838175e+09,8.134915e+09
2026-05-31,9.457125e+09,9.872631e+09,6.543928e+09,7.611682e+09,8.740841e+09,8.445241e+09,8.603216e+09
2026-06-30,1.037007e+10,1.043826e+10,7.255572e+09,7.402547e+09,9.374732e+09,8.968237e+09,9.049116e+09


In [5]:
from save_trade_forecast_long import to_long_format

# out: index=date, columns=예측치들
long_df = to_long_format(out, hs_code="854232", forecast_date=None)  # 오늘 날짜가 들어감
print(long_df.head())


           hs_code        indicator         value forecast_date
date                                                           
2025-10-31  854232      avg5_expDlr  8.071005e+09    2025-10-31
2025-10-31  854232  ensemble_expDlr  8.476852e+09    2025-10-31
2025-10-31  854232       ets_expDlr  8.606155e+09    2025-10-31
2025-10-31  854232      lstm_expDlr  8.201339e+09    2025-10-31
2025-10-31  854232   prophet_expDlr  6.235488e+09    2025-10-31


In [6]:
from save_long_forecast_to_db import save_long_forecast_to_db

# 이미 long_df가 만들어진 상태
save_long_forecast_to_db(
    long_df=long_df,
    db_info=db_info,  # {'host':..., 'port':..., 'user':..., 'password':..., 'database':...}
)

[INFO] 168 rows saved into 'korea_monthly_trade_forecast_v2' (Upsert by forecast_date).
